In [1]:
import os
import torch
import random
from PIL import Image, ImageFile
from collections import defaultdict

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMG_ROOT = "/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset"

ImageFile.LOAD_TRUNCATED_IMAGES = True

In [2]:
print(os.listdir("/kaggle/input/datasets/abhishekbuddiga06/embryo-dataset/embryo_dataset/embryo_dataset")[:10])

['FM1017-5', 'HC459-6', 'GJ191-1', 'LMMG218-1-10', 'RD1142-2', 'LS1035-1', 'GS826-2', 'PI1027-1', 'ZS435-5', 'PJ533-8']


In [3]:
def extract_label_from_folder(folder_name):
    try:
        return int(folder_name.split("-")[-1])
    except:
        return 0


def create_records():
    records = []

    for video_folder in os.listdir(IMG_ROOT):
        video_path = os.path.join(IMG_ROOT, video_folder)

        if not os.path.isdir(video_path):
            continue

        label = extract_label_from_folder(video_folder)

        for img_name in os.listdir(video_path):
            if img_name.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                img_path = os.path.join(video_path, img_name)
                records.append((img_path, label, video_folder))

    return records


records = create_records()
print("Total frames:", len(records))

Total frames: 342363


In [4]:
print(DEVICE)

cuda


In [5]:
def select_videos(records, num_videos=150):
    video_ids = list(set([r[2] for r in records]))
    selected = random.sample(video_ids, num_videos)
    return [r for r in records if r[2] in selected]


records = select_videos(records, 150)

labels = sorted(list(set([r[1] for r in records])))
label_map = {label: idx for idx, label in enumerate(labels)}

records = [(p, label_map[l], v) for (p, l, v) in records]

print("Labels:", set([r[1] for r in records]))

Labels: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10}


In [6]:
def split_by_video(records, train_ratio=0.7, val_ratio=0.15):
    video_ids = list(set([r[2] for r in records]))
    random.shuffle(video_ids)

    n = len(video_ids)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_vids = set(video_ids[:train_end])
    val_vids = set(video_ids[train_end:val_end])
    test_vids = set(video_ids[val_end:])

    train_records = [r for r in records if r[2] in train_vids]
    val_records = [r for r in records if r[2] in val_vids]
    test_records = [r for r in records if r[2] in test_vids]

    return train_records, val_records, test_records


train_records, val_records, test_records = split_by_video(records)

print(len(train_records), len(val_records), len(test_records))

52268 11233 11668


In [7]:
SEQ_LEN = 6

def create_sequences(records):
    video_dict = defaultdict(list)

    for path, label, vid in records:
        video_dict[vid].append((path, label))

    sequences = []

    for vid in video_dict:
        frames = sorted(video_dict[vid], key=lambda x: x[0])

        for i in range(len(frames) - SEQ_LEN + 1):
            seq = frames[i:i+SEQ_LEN]
            img_paths = [x[0] for x in seq]
            label = seq[-1][1]
            sequences.append((img_paths, label))

    return sequences


train_seq = create_sequences(train_records)
val_seq = create_sequences(val_records)
test_seq = create_sequences(test_records)

print(len(train_seq), len(val_seq), len(test_seq))

51743 11123 11553


In [8]:
transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor()
])


class EmbryoDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        paths, label = self.sequences[idx]

        imgs = []
        for p in paths:
            try:
                img = Image.open(p).convert("RGB")
                img = transform(img)
            except:
                img = torch.zeros(3, 128, 128)

            imgs.append(img)

        return torch.stack(imgs), torch.tensor(label)

In [9]:
train_loader = DataLoader(EmbryoDataset(train_seq), batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(EmbryoDataset(val_seq), batch_size=4, shuffle=False, num_workers=2)
test_loader = DataLoader(EmbryoDataset(test_seq), batch_size=4, shuffle=False, num_workers=2)

In [10]:
class CNN_LSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.cnn = models.resnet18(pretrained=True)
        self.cnn.fc = nn.Identity()

        self.lstm = nn.LSTM(512, 256, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.shape

        x = x.view(B*T, C, H, W)
        feats = self.cnn(x)

        feats = feats.view(B, T, -1)
        lstm_out, _ = self.lstm(feats)

        return self.fc(lstm_out[:, -1, :])

In [11]:
NUM_CLASSES = len(set([r[1] for r in records]))

model = CNN_LSTM(NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 200MB/s]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [12]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            preds = out.argmax(1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [13]:
def train(model, train_loader, val_loader, epochs=5):
    losses, accs = [], []

    for ep in range(epochs):
        model.train()
        total_loss = 0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        val_acc = evaluate(model, val_loader)

        losses.append(total_loss / len(train_loader))
        accs.append(val_acc)

        print(f"Epoch {ep+1}")
        print(f"Loss: {losses[-1]:.4f}")
        print(f"Val Acc: {val_acc:.4f}")

    return losses, accs


losses, accs = train(model, train_loader, val_loader, epochs=8)

Epoch 1
Loss: 0.1046
Val Acc: 0.8016
Epoch 2
Loss: 0.0240
Val Acc: 0.7139
Epoch 3
Loss: 0.0136
Val Acc: 0.5980
Epoch 4
Loss: 0.0098
Val Acc: 0.6861
Epoch 5
Loss: 0.0056
Val Acc: 0.7208
Epoch 6
Loss: 0.0075
Val Acc: 0.6762
Epoch 7
Loss: 0.0058
Val Acc: 0.6489
Epoch 8
Loss: 0.0042
Val Acc: 0.5940


In [14]:
test_acc = evaluate(model, test_loader)
print("Final Test Accuracy:", test_acc)

Final Test Accuracy: 0.6361118324244784


In [15]:
torch.save(model.state_dict(), "/kaggle/working/lstm_embryo.pth")

In [16]:
from IPython.display import FileLink
FileLink("lstm_embryo.pth")

/kaggle/working/lstm_embryo.pth

In [17]:
def count_parameters(model):
    print("Total:", sum(p.numel() for p in model.parameters()))
    print("Trainable:", sum(p.numel() for p in model.parameters() if p.requires_grad))

count_parameters(model)

Total: 11967819
Trainable: 11967819


In [18]:
from sklearn.metrics import classification_report

def detailed_evaluation(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            preds = out.argmax(1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    print(classification_report(all_labels, all_preds))

In [19]:
detailed_evaluation(model, test_loader)

              precision    recall  f1-score   support

           0       1.00      0.93      0.96      1103
           1       0.71      0.91      0.80      1441
           2       1.00      0.44      0.61      1688
           3       0.85      1.00      0.92      1667
           4       0.61      1.00      0.76       560
           5       0.39      0.64      0.48       550
           6       0.39      0.79      0.52       946
           7       0.73      0.50      0.59      1110
           8       0.40      0.26      0.32      1446
           9       0.00      0.00      0.00       561
          10       0.00      0.00      0.00       481

    accuracy                           0.64     11553
   macro avg       0.55      0.59      0.54     11553
weighted avg       0.65      0.64      0.61     11553

